In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords, movie_reviews
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. DATA UNDERSTANDING & LOADING
# Download necessary NLTK data
nltk.download('movie_reviews')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load the dataset into a DataFrame
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

df = pd.DataFrame(documents, columns=['review_tokens', 'sentiment'])
df['review_text'] = df['review_tokens'].apply(lambda x: " ".join(x))

print(f"Dataset Loaded. Shape: {df.shape}")
print(df['sentiment'].value_counts())

# ---
# 2. NLP PREPROCESSING
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercasing & removing special characters/numbers/URLs
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenization & Stopword removal & Lemmatization
    tokens = text.split()
    cleaned = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned)

print("Preprocessing text... (this may take a moment)")
df['clean_text'] = df['review_text'].apply(clean_text)

# ---
# 3. FEATURE ENGINEERING (TF-IDF)
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ---
# 4. MODEL BUILDING & 5. EVALUATION
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=20) # Depth limited to prevent heavy overfitting
}

results = []

for name, model in models.items():
    # Training
    model.fit(X_train_tfidf, y_train)

    # Prediction
    y_pred = model.predict(X_test_tfidf)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label='pos')
    recall = recall_score(y_test, y_pred, pos_label='pos')
    f1 = f1_score(y_test, y_pred, pos_label='pos')

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

# Display Comparison Table
results_df = pd.DataFrame(results)
print("\n--- MODEL COMPARISON ---")
print(results_df.sort_values(by="F1-Score", ascending=False))

# Optional: Detailed Report for the best model
print("\n--- Detailed Classification Report (Logistic Regression) ---")
print(classification_report(y_test, models["Logistic Regression"].predict(X_test_tfidf)))

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Dataset Loaded. Shape: (2000, 3)
sentiment
neg    1000
pos    1000
Name: count, dtype: int64
Preprocessing text... (this may take a moment)

--- MODEL COMPARISON ---
                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression    0.8125   0.815000  0.810945  0.812968
1          Naive Bayes    0.8050   0.836066  0.761194  0.796875
2        Decision Tree    0.6500   0.640553  0.691542  0.665072

--- Detailed Classification Report (Logistic Regression) ---
              precision    recall  f1-score   support

         neg       0.81      0.81      0.81       199
         pos       0.81      0.81      0.81       201

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400

